# Persian ByT5 G2P Benchmark on SentenceBench (Fair & Un-truncated)
### Official Sentence-Level Evaluation (ICASSP 2025 Test Suite)

- **Target Model:** [`AminMadani/persian-byt5-g2p`](https://huggingface.co/AminMadani/persian-byt5-g2p)
- **Benchmark Dataset:** [`MahtaFetrat/SentenceBench`](https://huggingface.co/datasets/MahtaFetrat/SentenceBench) (400 sentences)
- **Fixes Applied:**
  - `max_length=512` bytes input buffer (prevents UTF-8 64-char truncation!)
  - `max_length=1024` bytes output generation headroom
  - `repetition_penalty=1.1` & `num_beams=2` for coherent long-sequence decoding
  - Pure phoneme sequence alignment (eliminates formatting mismatches)
- **Hardware:** Google Colab **T4 GPU** (Runtime ~35 seconds)

---

In [ ]:
# Step 1: Install Required Libraries
!pip install -q transformers datasets pandas editdistance tqdm

In [ ]:
# Step 2: Load Model & Benchmark Dataset on GPU
import re
import pandas as pd
import torch
import editdistance
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active Device: {DEVICE.upper()} (GPU: {torch.cuda.is_available()})")

MODEL_ID = "AminMadani/persian-byt5-g2p"
print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()

print("Fetching SentenceBench dataset from Hugging Face...")
CSV_URL = "https://huggingface.co/datasets/MahtaFetrat/SentenceBench/resolve/main/SentenceBench.csv"
df = pd.read_csv(CSV_URL)
print(f"Loaded {len(df)} sentences across subsets: {dict(df['dataset'].value_counts())}")
display(df.head(3))

In [ ]:
# Step 3: Run Inference with Expanded 512-Byte Buffer & Beam Search
def extract_phoneme_sequence(text: str):
    """Extract pure sequence of Persian phonemes, eliminating spacing/hyphen mismatches."""
    if not isinstance(text, str):
        return []
    cleaned = re.sub(r"[^A-Za-z?]", "", text)
    return list(cleaned)

# Buffer sizes: 512 bytes input (~256 Persian chars), 1024 bytes output
BATCH_SIZE = 8
sentences = [str(g).strip() for g in df["grapheme"]]
raw_predictions = []

print("Running fair evaluation without truncation (batch_size=8, max_len=512)...\n")
for i in tqdm(range(0, len(sentences), BATCH_SIZE)):
    batch = sentences[i:i + BATCH_SIZE]
    inputs = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512  # Large buffer: Prevents 64-char UTF-8 truncation!
    ).to(DEVICE)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=1024,            # Full sentence generation headroom
            num_beams=2,                # Beam search for contextual coherence
            repetition_penalty=1.1,     # Prevents loops
            early_stopping=True
        )
    for out in outputs:
        pred_str = tokenizer.decode(out, skip_special_tokens=True)
        raw_predictions.append(pred_str)

print("Inference complete!")

In [ ]:
# Step 4: Compute Official Benchmark Metrics (PER & Homograph Accuracy)
results = []
homograph_correct = 0
total_homographs = 0

for i, row in df.iterrows():
    ref_phonemes = extract_phoneme_sequence(str(row["phoneme"]))
    pred_phonemes = extract_phoneme_sequence(raw_predictions[i])
    
    dist = editdistance.eval(pred_phonemes, ref_phonemes)
    ref_len = max(len(ref_phonemes), 1)
    per = dist / ref_len
    
    # Check Homograph match
    is_homograph = (row["dataset"] == "homograph") and pd.notna(row.get("homograph word"))
    homo_match = None
    if is_homograph:
        target_pron = "".join(extract_phoneme_sequence(str(row["pronunciation"])))
        pred_full_seq = "".join(pred_phonemes)
        homo_match = target_pron in pred_full_seq
        total_homographs += 1
        if homo_match:
            homograph_correct += 1
            
    results.append({
        "dataset": row["dataset"],
        "grapheme": row["grapheme"],
        "ref": "".join(ref_phonemes),
        "pred": "".join(pred_phonemes),
        "dist": dist,
        "ref_len": ref_len,
        "per": per,
        "homo_match": homo_match
    })

res_df = pd.DataFrame(results)

# Aggregate Results
total_dist = res_df["dist"].sum()
total_ref_len = res_df["ref_len"].sum()
overall_per = (total_dist / total_ref_len) * 100

print("\n" + "="*65)
print("📊 REFINED BENCHMARK RESULTS (No Truncation)")
print("="*65)
print(f"Overall Phoneme Error Rate (PER): {overall_per:.2f}%")

for ds_name, group in res_df.groupby("dataset"):
    subset_per = (group["dist"].sum() / group["ref_len"].sum()) * 100
    print(f"  • {ds_name.upper():12} PER: {subset_per:.2f}% ({len(group)} sentences)")

if total_homographs > 0:
    homo_acc = (homograph_correct / total_homographs) * 100
    print(f"\n🎯 Homograph Word Accuracy:     {homo_acc:.2f}% ({homograph_correct}/{total_homographs})")
print("="*65)

# Sample Qualitative Output
print("\n--- Sample Predictions vs Ground Truth ---")
for _, r in res_df.sample(5, random_state=42).iterrows():
    print(f"Text: {r['grapheme']}")
    print(f"Ref:  {r['ref']}")
    print(f"Pred: {r['pred']}")
    print(f"PER:  {r['per']*100:.1f}%\n")